In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import cv2
import os
import glob

def crop_dataset(img_dir, label_dir, out_img_dir, out_label_dir, top_ratio=0.45, bottom_ratio=0.15):
    os.makedirs(out_img_dir, exist_ok=True)
    os.makedirs(out_label_dir, exist_ok=True)

    img_paths = glob.glob(os.path.join(img_dir, '*.jpg'))
    print("Start")

    for img_path in img_paths:
        filename = os.path.basename(img_path)
        label_name = filename.replace('.jpg', '.txt')
        label_path = os.path.join(label_dir, label_name)

        img = cv2.imread(img_path)
        if img is None: continue
        h, w, _ = img.shape

        top_px = int(h * top_ratio)
        bottom_px = int(h * (1 - bottom_ratio))
        new_h = bottom_px - top_px

        cropped_img = img[top_px:bottom_px, :]
        cv2.imwrite(os.path.join(out_img_dir, filename), cropped_img)

        if not os.path.exists(label_path):
            continue

        with open(label_path, 'r') as f:
            lines = f.readlines()

        new_lines = []
        for line in lines:
            parts = line.strip().split()
            class_id = parts[0]
            cx, cy, bw, bh = map(float, parts[1:])

            abs_cy = cy * h
            abs_bh = bh * h

            box_top = abs_cy - (abs_bh / 2)
            box_bottom = abs_cy + (abs_bh / 2)

            if box_bottom < top_px or box_top > bottom_px:
                continue

            new_box_top = max(0, box_top - top_px)
            new_box_bottom = min(new_h, box_bottom - top_px)

            new_abs_h = new_box_bottom - new_box_top
            new_abs_cy = new_box_top + (new_abs_h / 2)

            new_cx = cx
            new_cy = new_abs_cy / new_h
            new_bw = bw
            new_bh = new_abs_h / new_h

            new_lines.append(f"{class_id} {new_cx:.6f} {new_cy:.6f} {new_bw:.6f} {new_bh:.6f}\n")

        with open(os.path.join(out_label_dir, label_name), 'w') as f:
            f.writelines(new_lines)

    print("done")

ORIGINAL_IMG_DIR = '/content/dataset_yolo/train/images'
ORIGINAL_LABEL_DIR = '/content/dataset_yolo/train/labels'

NEW_IMG_DIR = '/content/dataset_cropped/train/images'
NEW_LABEL_DIR = '/content/dataset_cropped/train/labels'

crop_dataset(ORIGINAL_IMG_DIR, ORIGINAL_LABEL_DIR, NEW_IMG_DIR, NEW_LABEL_DIR)

Start
done


In [7]:
import cv2
import os
import glob
import numpy as np

def imread_korean(path):
    img_array = np.fromfile(path, np.uint8)
    return cv2.imdecode(img_array, cv2.IMREAD_COLOR)

def imwrite_korean(path, img):
    ext = os.path.splitext(path)[1]
    result, n = cv2.imencode(ext, img)
    if result:
        with open(path, mode='w+b') as f:
            n.tofile(f)

def crop_dataset(img_dir, label_dir, out_img_dir, out_label_dir, top_ratio=0.45, bottom_ratio=0.15):
    os.makedirs(out_img_dir, exist_ok=True)
    os.makedirs(out_label_dir, exist_ok=True)

    print(f"코드에서 찾고 있는 폴더: {img_dir}")

    img_paths = []
    for ext in ('*.jpg', '*.jpeg', '*.png', '*.JPG', '*.JPEG', '*.PNG'):
        img_paths.extend(glob.glob(os.path.join(img_dir, ext)))

    if len(img_paths) == 0:
        print("경로 복사를 다시 확인해주세요!")
        return

    print(f"{len(img_paths)}장의 이미지")

    for img_path in img_paths:
        filename = os.path.basename(img_path)
        label_name = os.path.splitext(filename)[0] + '.txt'
        label_path = os.path.join(label_dir, label_name)

        img = imread_korean(img_path)
        if img is None: continue
        h, w, _ = img.shape

        top_px = int(h * top_ratio)
        bottom_px = int(h * (1 - bottom_ratio))
        new_h = bottom_px - top_px

        cropped_img = img[top_px:bottom_px, :]
        imwrite_korean(os.path.join(out_img_dir, filename), cropped_img)

        if not os.path.exists(label_path):
            continue

        with open(label_path, 'r', encoding='utf-8') as f:
            lines = f.readlines()

        new_lines = []
        for line in lines:
            parts = line.strip().split()
            class_id = parts[0]
            cx, cy, bw, bh = map(float, parts[1:])

            abs_cy = cy * h
            abs_bh = bh * h
            box_top = abs_cy - (abs_bh / 2)
            box_bottom = abs_cy + (abs_bh / 2)

            if box_bottom < top_px or box_top > bottom_px:
                continue

            new_box_top = max(0, box_top - top_px)
            new_box_bottom = min(new_h, box_bottom - top_px)

            new_abs_h = new_box_bottom - new_box_top
            new_abs_cy = new_box_top + (new_abs_h / 2)

            new_cx = cx
            new_cy = new_abs_cy / new_h
            new_bw = bw
            new_bh = new_abs_h / new_h

            new_lines.append(f"{class_id} {new_cx:.6f} {new_cy:.6f} {new_bw:.6f} {new_bh:.6f}\n")

        with open(os.path.join(out_label_dir, label_name), 'w', encoding='utf-8') as f:
            f.writelines(new_lines)

    print("🎉 크롭 완료! 왼쪽 폴더 탭을 새로고침 해보세요!")


ORIGINAL_IMG_DIR = '/content/drive/MyDrive/dataset_yolo/train/images'
ORIGINAL_LABEL_DIR = '/content/drive/MyDrive/dataset_yolo/train/labels'

NEW_IMG_DIR = '/content/drive/MyDrive/dataset_cropped/train/images'
NEW_LABEL_DIR = '/content/drive/MyDrive/dataset_cropped/train/labels'

crop_dataset(ORIGINAL_IMG_DIR, ORIGINAL_LABEL_DIR, NEW_IMG_DIR, NEW_LABEL_DIR)

코드에서 찾고 있는 폴더: /content/drive/MyDrive/dataset_yolo/train/images
1607장의 이미지
🎉 크롭 완료! 왼쪽 폴더 탭을 새로고침 해보세요!


In [8]:
import cv2
import os
import glob
import numpy as np

def imread_korean(path):
    img_array = np.fromfile(path, np.uint8)
    return cv2.imdecode(img_array, cv2.IMREAD_COLOR)

def imwrite_korean(path, img):
    ext = os.path.splitext(path)[1]
    result, n = cv2.imencode(ext, img)
    if result:
        with open(path, mode='w+b') as f:
            n.tofile(f)

def crop_dataset(img_dir, label_dir, out_img_dir, out_label_dir, top_ratio=0.45, bottom_ratio=0.15):
    os.makedirs(out_img_dir, exist_ok=True)
    os.makedirs(out_label_dir, exist_ok=True)

    print(f"코드에서 찾고 있는 폴더: {img_dir}")

    img_paths = []
    for ext in ('*.jpg', '*.jpeg', '*.png', '*.JPG', '*.JPEG', '*.PNG'):
        img_paths.extend(glob.glob(os.path.join(img_dir, ext)))

    if len(img_paths) == 0:
        print("경로 복사를 다시 확인해주세요!")
        return

    print(f"{len(img_paths)}장의 이미지")

    for img_path in img_paths:
        filename = os.path.basename(img_path)
        label_name = os.path.splitext(filename)[0] + '.txt'
        label_path = os.path.join(label_dir, label_name)

        img = imread_korean(img_path)
        if img is None: continue
        h, w, _ = img.shape

        top_px = int(h * top_ratio)
        bottom_px = int(h * (1 - bottom_ratio))
        new_h = bottom_px - top_px

        cropped_img = img[top_px:bottom_px, :]
        imwrite_korean(os.path.join(out_img_dir, filename), cropped_img)

        if not os.path.exists(label_path):
            continue

        with open(label_path, 'r', encoding='utf-8') as f:
            lines = f.readlines()

        new_lines = []
        for line in lines:
            parts = line.strip().split()
            class_id = parts[0]
            cx, cy, bw, bh = map(float, parts[1:])

            abs_cy = cy * h
            abs_bh = bh * h
            box_top = abs_cy - (abs_bh / 2)
            box_bottom = abs_cy + (abs_bh / 2)

            if box_bottom < top_px or box_top > bottom_px:
                continue

            new_box_top = max(0, box_top - top_px)
            new_box_bottom = min(new_h, box_bottom - top_px)

            new_abs_h = new_box_bottom - new_box_top
            new_abs_cy = new_box_top + (new_abs_h / 2)

            new_cx = cx
            new_cy = new_abs_cy / new_h
            new_bw = bw
            new_bh = new_abs_h / new_h

            new_lines.append(f"{class_id} {new_cx:.6f} {new_cy:.6f} {new_bw:.6f} {new_bh:.6f}\n")

        with open(os.path.join(out_label_dir, label_name), 'w', encoding='utf-8') as f:
            f.writelines(new_lines)

    print("크롭 완료")


ORIGINAL_IMG_DIR = '/content/drive/MyDrive/dataset_yolo/val/images'
ORIGINAL_LABEL_DIR = '/content/drive/MyDrive/dataset_yolo/val/labels'

NEW_IMG_DIR = '/content/drive/MyDrive/dataset_cropped/val/images'
NEW_LABEL_DIR = '/content/drive/MyDrive/dataset_cropped/val/labels'

crop_dataset(ORIGINAL_IMG_DIR, ORIGINAL_LABEL_DIR, NEW_IMG_DIR, NEW_LABEL_DIR)

코드에서 찾고 있는 폴더: /content/drive/MyDrive/dataset_yolo/val/images
402장의 이미지
크롭 완료
